# Controlled full-covariance GMM check

Test whether a full covariance matrix per component improves on the current tied-covariance GMM candidate. Change **only covariance type** at 10 components. Keep 150 embeddings, training-only scalers, random seed 42, 3 initializations, covariance regularization 0.001, EM tolerance 0.001, maximum 500 iterations, the partial-label rule, and logistic regression C=1000 fixed.

Reuse the classifier and labeling helpers from the method-comparison experiment. Fit GMM only on the 1,171 training trials, estimate reason prevalences only from the 157 known labels, and evaluate on the same 78 validation trials. No additional PCA, year, or other features are introduced. This offline test uses no Kaggle submission. Keep previous datasets, models, submissions, and tags unchanged.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
K_VALUES = [10]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
GMM_RULE = {'n_init': 3, 'max_iter': 500, 'tol': 0.001, 'reg_covar': 0.001, 'random_state': 42, 'init_params': 'kmeans'}
CLASSIFIER_GRID = [("logistic", 1000.0)]
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"full_covariance_check"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


## Shared label-generation, training, and scoring rules

The following helpers are copied from the preceding method-comparison notebook. Unknown targets remain -100, original labels remain unchanged, generated weights are capped per reason/class, and logistic fitting weights sum to 157 per reason. The zero-augmentation supervised control must reproduce its saved probabilities.

In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"]=="logistic":
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
tied_path=DATA/"experiments/method_comparison/classifier_models/gmm_tied_k10__logistic_C1000.joblib"
tied_dataset_path=DATA/"experiments/method_comparison/datasets/gmm_tied_k10.csv"
tied_model=joblib.load(tied_path)
tied_probabilities=predict_reason_models(tied_model,X_valid)
tied_metrics=save_predictions("gmm_tied_k10_reference",tied_probabilities)
prior_results=pd.read_csv(DATA/"experiments/method_comparison/classifier_comparison.csv").set_index("experiment")
np.testing.assert_allclose(tied_metrics["sample_f1"],prior_results.loc["gmm_tied_k10__logistic_C1000","sample_f1"],atol=1e-14)
protected_hashes.update({str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in [tied_path,tied_dataset_path]})

with threadpool_limits(limits=1):
    full_clustering=GaussianMixture(n_components=10,covariance_type="full",**GMM_RULE)
    with warnings.catch_warnings(record=True) as full_warnings:
        warnings.simplefilter("always",ConvergenceWarning)
        full_clustering.fit(XC)
    assert full_clustering.converged_, "Full-covariance GMM failed to converge; do not rank this fit."
    responsibilities=full_clustering.predict_proba(XC)
    targets,weights,estimated,support,components=partial_labels(responsibilities)
    full_dataset=save_dataset("gmm_full_k10",targets,weights,estimated,responsibilities.argmax(axis=1),support)
    components.to_csv(OUT/"clusters/gmm_full_k10.csv",index=False)
    full_model=fit_reason_models("gmm_full_k10","logistic",1000.0)
    full_probabilities=predict_reason_models(full_model,X_valid)
    full_metrics=save_predictions("gmm_full_k10",full_probabilities)
    joblib.dump({"clustering":full_clustering,"scaler":cluster_scaler,"label_rule":LABEL_RULE},OUT/"clustering_models/gmm_full_k10.joblib",compress=3)
    joblib.dump(full_model,OUT/"classifier_models/gmm_full_k10__logistic_C1000.joblib",compress=3)
    eigenvalues=np.linalg.eigvalsh(full_clustering.covariances_)
comparison=pd.DataFrame([
    {"experiment":"gmm_tied_k10","covariance":"tied","covariance_parameters":11325,
     "generated_trials":int(pd.read_csv(tied_dataset_path).loc[lambda d:(d[[f'weight_{r}' for r in REASONS]]>0).any(axis=1)&~known].shape[0]),**tied_metrics},
    {"experiment":"gmm_full_k10","covariance":"full","covariance_parameters":113250,
     "generated_trials":full_dataset["generated_trials"],**full_metrics},
])
comparison.to_csv(OUT/"comparison.csv",index=False)
pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
pd.DataFrame({"component":np.arange(10),"minimum_covariance_eigenvalue":eigenvalues.min(axis=1),
              "maximum_covariance_eigenvalue":eigenvalues.max(axis=1)}).to_csv(OUT/"covariance_diagnostics.csv",index=False)
print("Full GMM converged after",full_clustering.n_iter_,"iterations.")
display(comparison[["experiment","generated_trials","sample_f1","macro_f1","fallback_trials"]].round(4))
display(comparison.set_index("experiment")[[f"f1_{r}" for r in REASONS]].round(4))

Full GMM converged after 2 iterations.


,experiment,generated_trials,sample_f1,macro_f1,fallback_trials
0,gmm_tied_k10,696,0.9060,0.8116,4
1,gmm_full_k10,922,0.8675,0.7624,2


,f1_Safety,f1_Funding,f1_Efficacy,f1_Recruitment,f1_Other
experiment,,,,,
gmm_tied_k10,0.9412,0.6667,0.8387,0.9800,0.6316
gmm_full_k10,0.9412,0.6667,0.6429,0.9615,0.6000


In [4]:
# Replay the full-covariance classifier from its exact saved dataset.
frame=pd.read_csv(OUT/"datasets/gmm_full_k10.csv",float_precision="round_trip")
assert frame.ID.tolist()==train.ID.tolist()
replay={**full_model,"estimators":[]}
saved_scaled=classifier_scaler.transform(frame[FEATURES].to_numpy())
with threadpool_limits(limits=1):
    for r in REASONS:
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy()
        include=w>0
        estimator=LogisticRegression(C=1000.0,solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(saved_scaled[include],y[include],sample_weight=w[include]*known.sum()/w[include].sum())
        replay["estimators"].append(estimator)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(full_probabilities)[0])
loaded=joblib.load(OUT/"classifier_models/gmm_full_k10__logistic_C1000.joblib")
np.testing.assert_allclose(predict_reason_models(loaded,X_valid),full_probabilities,rtol=1e-12,atol=1e-12)
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

change=full_metrics["sample_f1"]-tied_metrics["sample_f1"]
candidate_path=None
if change>1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv")
    sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    predicted,empty=label_predictions(predict_reason_models(loaded,test[FEATURES].to_numpy()))
    submission=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    submission=submission.set_index("ID").loc[sample.ID].reset_index()
    assert submission.shape==(312,2) and submission.Predicted.str.len().gt(0).all()
    candidate_path=OUT/"submissions/part1_gmm_full_k10_logistic_C1000_submission.csv"
    submission.to_csv(candidate_path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(candidate_path),submission)

record={"component_count":10,"changed_setting":"covariance_type: tied to full","gmm_rule":GMM_RULE,
    "label_rule":LABEL_RULE,"classifier_C":1000.0,"features":FEATURES,"threshold":0.5,"nonempty_fallback":True,
    "full_converged":bool(full_clustering.converged_),"full_iterations":int(full_clustering.n_iter_),
    "full_bic":float(full_clustering.bic(XC)),"minimum_full_covariance_eigenvalue":float(eigenvalues.min()),
    "tied_validation_sample_f1":tied_metrics["sample_f1"],"full_validation_sample_f1":full_metrics["sample_f1"],
    "validation_change":change,"protected_input_sha256":protected_hashes,
    "dataset_sha256":hashlib.sha256((OUT/"datasets/gmm_full_k10.csv").read_bytes()).hexdigest(),
    "candidate_submission":None if candidate_path is None else str(candidate_path.relative_to(OUT)),
    "candidate_submission_sha256":None if candidate_path is None else hashlib.sha256(candidate_path.read_bytes()).hexdigest(),
    "kaggle_score":None,"current_reported_kaggle_best":0.87393,
}
(OUT/"settings.json").write_text(json.dumps(record,indent=2)+"\n")
lines=["# Full versus tied covariance: controlled K=10 check","",
       "Only GMM covariance type changes. Both methods use 150 embeddings, the same training-only scalers, "
       "3 initializations, covariance regularization 0.001, maximum 500 EM iterations, partial-label rules, "
       "logistic regression C=1000, threshold 0.5, and nonempty fallback.","",
       "| Covariance | Generated trials | Sample-wise F1 | Macro F1 | Covariance parameters |",
       "|---|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):lines.append(f"| {row.covariance} | {row.generated_trials} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.covariance_parameters} |")
lines.extend(["",f"Full covariance converged after **{full_clustering.n_iter_}** iterations. "
              f"Its validation sample-wise F1 difference from tied covariance is **{change:+.4f}**.","",
              "## Per-reason F1","","| Covariance | Safety | Funding | Efficacy | Recruitment | Other |","|---|---:|---:|---:|---:|---:|"])
for row in comparison.to_dict('records'):
    scores=" | ".join(f"{row[f'f1_{r}']:.4f}" for r in REASONS)
    lines.append(f"| {row['covariance']} | {scores} |")
lines.extend(["","The extra covariance parameters can improve or harm the label geometry; the validation result is the evidence "
              "for this particular setting. It does not establish a general rule about full covariance. "
              "Neither covariance candidate has a measured Kaggle score. The current reported best remains 0.87393.","",
              "`datasets/`, `label_audits/`, and `clusters/` preserve the exact augmented data and estimates. "
              "`clustering_models/` and `classifier_models/` preserve the fitted models. `validation_predictions/` preserves "
              "the compared predictions. `covariance_diagnostics.csv` records eigenvalue ranges. "
              "`settings.json` records configuration and checksums. Refit and reload checks reproduce the selected full model; "
              "all previous input/model/submission files remain unchanged.",""])
if candidate_path is not None:lines.append(f"Candidate file prepared for optional manual upload: `{candidate_path.relative_to(OUT)}`. No Kaggle slot has been used.")
else:lines.append("Full covariance did not improve the primary validation metric; retain the tied-covariance candidate for the remaining Kaggle submission.")
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)

# Full versus tied covariance: controlled K=10 check

Only GMM covariance type changes. Both methods use 150 embeddings, the same training-only scalers, 3 initializations, covariance regularization 0.001, maximum 500 EM iterations, partial-label rules, logistic regression C=1000, threshold 0.5, and nonempty fallback.

| Covariance | Generated trials | Sample-wise F1 | Macro F1 | Covariance parameters |
|---|---:|---:|---:|---:|
| tied | 696 | 0.9060 | 0.8116 | 11325 |
| full | 922 | 0.8675 | 0.7624 | 113250 |

Full covariance converged after **2** iterations. Its validation sample-wise F1 difference from tied covariance is **-0.0385**.

## Per-reason F1

| Covariance | Safety | Funding | Efficacy | Recruitment | Other |
|---|---:|---:|---:|---:|---:|
| tied | 0.9412 | 0.6667 | 0.8387 | 0.9800 | 0.6316 |
| full | 0.9412 | 0.6667 | 0.6429 | 0.9615 | 0.6000 |

The extra covariance parameters can improve or harm the label geometry; the validation result is the evidence for this particular 